# 11.4.2 FAISS Embedding 저장 및 검색

FAISS(Facebook AI Similarity Search)를 사용해 문서 embedding을 저장하고, 입력 query와 가장 유사한 문서를 검색하는 예제입니다.

원본 txt에는 `FASSIS`라고 적혀 있지만, 실제 라이브러리 이름은 `FAISS`입니다.

이 노트북의 흐름:

1. 샘플 문서 준비
2. OpenAI Embedding + FAISS 방식
3. 현재 환경 검증용 SentenceTransformer + NumPy 방식
4. 검색 결과 확인


## 1. 설치 확인

`faiss` 모듈이 없으면 FAISS 방식 셀은 실행되지 않습니다.

필요하면 터미널에서 다음 명령으로 설치합니다.

```powershell
uv add faiss-cpu
```

OpenAI Embedding을 쓰려면 `.env` 파일 또는 환경변수에 `OPENAI_API_KEY`가 있어야 합니다.

In [1]:
import importlib.util
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.documents import Document

try:
    from langchain_text_splitters import CharacterTextSplitter
except ImportError:
    from langchain.text_splitter import CharacterTextSplitter

load_dotenv()

has_faiss = importlib.util.find_spec("faiss") is not None
has_openai_key = bool(os.getenv("OPENAI_API_KEY"))

print("faiss installed:", has_faiss)
print("OPENAI_API_KEY set:", has_openai_key)

faiss installed: True
OPENAI_API_KEY set: True


## 2. 샘플 문서 준비

벡터, 차원, FAISS에 대한 짧은 설명 문서를 `Document` 객체로 만듭니다.

In [2]:
documents = [
    Document(
        page_content=(
            "벡터(vector)는 크기와 방향을 가진 수학적 개념으로, "
            "기하학과 물리학뿐 아니라 컴퓨터 과학과 머신러닝에서도 핵심적으로 사용됩니다."
        )
    ),
    Document(
        page_content=(
            "차원(dimension)은 수학, 물리학, 컴퓨터 과학에서 데이터의 위치와 구조, "
            "복잡성을 표현하기 위한 개념입니다."
        )
    ),
    Document(
        page_content=(
            "FAISS(Facebook AI Similarity Search)는 Meta AI에서 개발한 오픈소스 라이브러리로, "
            "대규모 벡터 데이터베이스에서 효율적인 벡터 검색과 유사도 검색을 수행하도록 최적화되어 있습니다."
        )
    ),
]

text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_documents = text_splitter.split_documents(documents)

print("document count:", len(documents))
print("split document count:", len(split_documents))

for idx, doc in enumerate(split_documents, start=1):
    print(f"Document {idx}:", doc.page_content)

document count: 3
split document count: 3
Document 1: 벡터(vector)는 크기와 방향을 가진 수학적 개념으로, 기하학과 물리학뿐 아니라 컴퓨터 과학과 머신러닝에서도 핵심적으로 사용됩니다.
Document 2: 차원(dimension)은 수학, 물리학, 컴퓨터 과학에서 데이터의 위치와 구조, 복잡성을 표현하기 위한 개념입니다.
Document 3: FAISS(Facebook AI Similarity Search)는 Meta AI에서 개발한 오픈소스 라이브러리로, 대규모 벡터 데이터베이스에서 효율적인 벡터 검색과 유사도 검색을 수행하도록 최적화되어 있습니다.


## 3. OpenAI Embedding + FAISS 검색

원본 txt와 가장 가까운 방식입니다.

이 셀은 `faiss-cpu` 설치와 `OPENAI_API_KEY` 설정이 모두 되어 있을 때 실행됩니다.

In [3]:
if not has_faiss:
    print("faiss가 설치되어 있지 않아 FAISS 셀을 건너뜁니다. 필요하면 `uv add faiss-cpu`를 실행하세요.")
elif not has_openai_key:
    print("OPENAI_API_KEY가 없어 OpenAI Embedding 셀을 건너뜁니다.")
else:
    from langchain_community.vectorstores import FAISS
    from langchain_openai import OpenAIEmbeddings

    embeddings = OpenAIEmbeddings()
    faiss_index = FAISS.from_documents(split_documents, embeddings)

    query = "What is FAISS?"
    results = faiss_index.similarity_search(query, k=3)

    print("Result:")
    print(results[0].page_content)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_5504\3812788609.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Result:
FAISS(Facebook AI Similarity Search)는 Meta AI에서 개발한 오픈소스 라이브러리로, 대규모 벡터 데이터베이스에서 효율적인 벡터 검색과 유사도 검색을 수행하도록 최적화되어 있습니다.


## 4. 현재 환경 검증용: SentenceTransformer + NumPy 검색

FAISS가 아직 설치되지 않은 환경에서도 embedding 검색 흐름을 확인할 수 있도록 로컬 모델과 NumPy로 유사도 검색을 수행합니다.

FAISS를 쓰는 것은 아니지만, `문서 embedding 생성 -> query embedding 생성 -> 가장 가까운 문서 검색`이라는 핵심 흐름은 같습니다.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer


def cosine_similarity_matrix(query_embedding, document_embeddings):
    query_norm = query_embedding / np.linalg.norm(query_embedding, axis=1, keepdims=True)
    docs_norm = document_embeddings / np.linalg.norm(document_embeddings, axis=1, keepdims=True)
    return query_norm @ docs_norm.T


local_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

texts = [doc.page_content for doc in split_documents]
document_embeddings = local_model.encode(texts)

query = "What is FAISS?"
query_embedding = local_model.encode([query])

scores = cosine_similarity_matrix(query_embedding, document_embeddings)[0]
ranked_indexes = np.argsort(scores)[::-1]

print("Query:", query)
print()

for rank, doc_index in enumerate(ranked_indexes, start=1):
    print(f"Rank {rank} / score={scores[doc_index]:.4f}")
    print(texts[doc_index])
    print("-" * 80)

## 5. FAISS 인덱스 저장 예시

`FAISS.from_documents(...)`로 만든 인덱스는 로컬 디렉터리에 저장할 수 있습니다.

아래 셀도 `faiss-cpu`와 `OPENAI_API_KEY`가 준비된 경우에만 실행됩니다.

In [4]:
if not has_faiss or not has_openai_key:
    print("FAISS 저장 예시는 faiss-cpu 설치와 OPENAI_API_KEY 설정 후 실행하세요.")
else:
    save_dir = Path("faiss_store_11_4_2")
    faiss_index.save_local(str(save_dir))
    print("saved:", save_dir.resolve())

saved: C:\Users\Playdata\study\agent-dev\04_vectordb\faiss_store_11_4_2


## 6. 정리

- FAISS는 대규모 벡터에서 빠르게 유사한 항목을 찾기 위한 라이브러리입니다.
- LangChain의 `FAISS.from_documents()`를 사용하면 문서 embedding과 인덱스 생성을 쉽게 연결할 수 있습니다.
- OpenAI Embedding을 사용하려면 API 키가 필요합니다.
- 현재 환경처럼 `faiss`가 없을 때는 먼저 `uv add faiss-cpu`로 설치한 뒤 FAISS 셀을 실행하면 됩니다.